**ÍNDICE DEL PROYECTO**
1. **Análisis y Exploración del Dataset**
   - 1.1 Carga de Matrices y Limpieza de Estructuras
   - 1.2 Mapeo de Entidades (CEDIS y Sucursales)
2. **Modelado del Problema Combinatorio (VRP / TSP)**
   - 2.1 Definición de la Función Objetivo y Cálculo de Línea Base
3. **Desarrollo e Implementación del Algoritmo Heurístico**
   - 3.1 Operador de Vecindad (Swap / 2-Opt)
   - 3.2 Algoritmo de Recocido Simulado (Simulated Annealing)
4. **Ejecución de Simulaciones y Registro de Métricas**
   - 4.1 Experimentos y Tracking de Costos
5. **Evaluación de Rendimiento y Comparativa**
   - 5.1 Análisis de Convergencia Visual
6. **Generación del Mapa Interactivo Geoespacial**
   - 6.1 Renderizado de la Ruta Optimizada sobre Culiacán

**INTRODUCCIÓN**

La distribución logística de insumos de abarrotes en Culiacán, Sinaloa, presenta retos operativos debido a la dispersión geográfica de las sucursales, patrones de tráfico urbano y variaciones en los costos de consumo de combustible. Probar todas las combinaciones de rutas de forma exhaustiva resulta inviable ya que el Problema de Enrutamiento de Vehículos (VRP) es de complejidad $NP$-hard.

En esta libreta se documenta el diseño, modelado e implementación computacional de una solución basada en la metaheurística de **Recocido Simulado (Simulated Annealing)** en Python. El objetivo es minimizar la distancia recorrida y los costos asociados de combustible desde un Centro de Distribución (CEDIS) hacia 90 sucursales de restaurantes, garantizando eficiencia logística, comparabilidad frente a una línea base sin optimizar y visualización geoespacial interactiva.

**1.- Análisis y Exploración del Dataset**

En este primer paso importamos las librerías necesarias y realizamos la lectura de los tres archivos base del proyecto:
- `datos_distribucion_tiendas.xlsx`: Información descriptiva de los 100 nodos (tipo, coordenadas, capacidades).
- `matriz_distancias.xlsx`: Matriz de $100 \times 100$ con las distancias inter-nodo en kilómetros.
- `matriz_costos_combustible.xlsx`: Matriz de $100 \times 100$ con los costos de consumo de combustible en MXN.

**1.1.- Carga de Matrices y Limpieza de Estructuras**

**Funciones y atributos utilizados:**
1. **`pd.read_excel()`**: Carga el contenido de los archivos de Excel en un DataFrame de Pandas.
2. **`.to_numpy()`**: Convierte las matrices bidimensionales de Pandas en arreglos matriciales de NumPy para optimizar sustancialmente la velocidad de cálculo aritmético durante las iteraciones del algoritmo.
3. **`.shape`**: Retorna una tupla con las dimensiones (filas, columnas) de los conjuntos de datos.
4. **`.value_counts()`**: Contabiliza la cantidad de registros por tipo de inmueble (Centro de Distribución vs. Tienda).

In [7]:
import numpy as np
import pandas as pd

# Lectura de los archivos Excel suministrados
df_tiendas = pd.read_excel('./Datasets/datos_distribucion_tiendas.xlsx')
df_distancias = pd.read_excel('./Datasets/matriz_distancias.xlsx')
df_costos = pd.read_excel('./Datasets/matriz_costos_combustible.xlsx')

# Conversión a arreglos bidimensionales de NumPy para alto rendimiento
matriz_dist = df_distancias.to_numpy()
matriz_costo = df_costos.to_numpy()

# Validación de dimensiones e integridad
print('=== VERIFICACIÓN DE ESTRUCTURAS DE DATOS===')
print(f'Dimensiones Matriz Distancias: {matriz_dist.shape} (Esperado: 100x100)')
print(f'Dimensiones Matriz Costos:{matriz_costo.shape} (Esperado: 100x100)')
print('\nDesglose de Entidades en el Dataset:')
print(df_tiendas['Tipo'].value_counts())

=== VERIFICACIÓN DE ESTRUCTURAS DE DATOS===
Dimensiones Matriz Distancias: (100, 100) (Esperado: 100x100)
Dimensiones Matriz Costos:(100, 100) (Esperado: 100x100)

Desglose de Entidades en el Dataset:
Tipo
Tienda                    90
Centro de Distribución    10
Name: count, dtype: int64


**1.2.- Mapeo de Entidades (CEDIS y Sucursales)**

**Funciones y atributos utilizados:**
1. **`df.index`**: Recupera los índices numéricos correspondientes a cada fila del DataFrame.
2. **`list(range())`**: Genera la secuencia numérica que representa las posiciones de las sucursales a entregar.
3. **`iloc[]`**: Permite acceder a un registro específico por su posición de índice.

Dividimos el dataset identificando las primeras 10 filas (índices 0 al 9) como **Centros de Distribución (CEDIS)** y los 90 nodos restantes (índices 10 al 99) como las **sucursales de venta / restaurantes**. Para la primera prueba elegiremos el **Centro de Distribución 1** (índice 0) como base de salida y retorno.

In [8]:
# Filtrado de entidades por su tipo
cedis_df = df_tiendas[df_tiendas['Tipo'] == 'Centro de Distribución']
tiendas_df = df_tiendas[df_tiendas['Tipo'] == 'Tienda']

# Seleccion de CEDIS origen y extracción deíndices de las sucursales a visitar
cedis_origen_idx = 0
sucursales_indices = list(range(len(cedis_df), len(df_tiendas)))

print(f"CEDIS DE ORIGEN SELECCIONADO: {df_tiendas.iloc[cedis_origen_idx]['Nombre']}")
print(f'TOTAL DE SUCURSALES A ENTREGAR: {len(sucursales_indices)}')

CEDIS DE ORIGEN SELECCIONADO: Centro de Distribución 1
TOTAL DE SUCURSALES A ENTREGAR: 90
